<a href="https://colab.research.google.com/github/juanepstein99/DI_Bootcamp/blob/main/Week18/Day2/MiniProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Brazilian E-Commerce — Dive Deep Using SQL

### Mini Project — Advanced SQL

This notebook analyzes the Brazilian E-Commerce dataset using **Pandas**, **SQLAlchemy**, and **SQLite**.

The project covers:

1. Data loading and exploration  
2. Creating a SQLite database  
3. January 2018 five-star review analysis  
4. Year-on-year customer purchase trends  
5. Customer average order value  
6. Top cities by revenue  
7. State-level revenue  
8. Seller performance  
9. Delivery success rate  
10. Preferred payment method by product category  
11. Distance between seller and customer cities  

The SQL queries were written specifically for **SQLite**.

## 1. Load and Explore the Data

In [1]:
# Import the required libraries.

import os
import glob
import zipfile
import numpy as np
import pandas as pd
import sqlite3

from sqlalchemy import create_engine

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Upload the ZIP file supplied with the project.
#
# In Google Colab, upload:
# "Brazilian E-Commerce Dive Deep using SQL.zip"

from google.colab import files

uploaded = files.upload()
zip_filename = list(uploaded.keys())[0]

extract_dir = "/content/brazilian_ecommerce"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

# The supplied ZIP contains some CSVs inside nested folders,
# so recursive search makes the loading process robust.
csv_paths = glob.glob(
    os.path.join(extract_dir, "**", "*.csv"),
    recursive=True
)

print(f"CSV files found: {len(csv_paths)}")

for path in sorted(csv_paths):
    print("-", os.path.basename(path))

Saving Brazilian E-Commerce Dive Deep using SQL.zip to Brazilian E-Commerce Dive Deep using SQL.zip
CSV files found: 16
- olist_customers_dataset.csv
- olist_customers_dataset.csv
- olist_geolocation_dataset.csv
- olist_geolocation_dataset.csv
- olist_order_items_dataset.csv
- olist_order_items_dataset.csv
- olist_order_payments_dataset.csv
- olist_order_payments_dataset.csv
- olist_order_reviews_dataset.csv
- olist_order_reviews_dataset.csv
- olist_orders_dataset.csv
- olist_orders_dataset.csv
- olist_products_dataset.csv
- olist_products_dataset.csv
- olist_sellers_dataset.csv
- product_category_name_translation.csv


In [3]:
# Create a dictionary that maps each CSV filename to its actual path.

csv_lookup = {
    os.path.basename(path): path
    for path in csv_paths
}

required_files = [
    "olist_customers_dataset.csv",
    "olist_sellers_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_products_dataset.csv",
    "olist_geolocation_dataset.csv",
    "product_category_name_translation.csv",
    "olist_orders_dataset.csv",
    "olist_order_payments_dataset.csv"
]

missing_files = [
    filename
    for filename in required_files
    if filename not in csv_lookup
]

if missing_files:
    raise FileNotFoundError(
        f"Missing expected files: {missing_files}"
    )

print("All required files were found.")

All required files were found.


In [4]:
# Load every CSV into a Pandas DataFrame.

df_olist_customers = pd.read_csv(
    csv_lookup["olist_customers_dataset.csv"]
)

df_olist_sellers = pd.read_csv(
    csv_lookup["olist_sellers_dataset.csv"]
)

df_olist_order_reviews = pd.read_csv(
    csv_lookup["olist_order_reviews_dataset.csv"]
)

df_olist_order_items = pd.read_csv(
    csv_lookup["olist_order_items_dataset.csv"]
)

df_olist_products = pd.read_csv(
    csv_lookup["olist_products_dataset.csv"]
)

df_olist_geolocation = pd.read_csv(
    csv_lookup["olist_geolocation_dataset.csv"]
)

df_product_category_name_translation = pd.read_csv(
    csv_lookup["product_category_name_translation.csv"]
)

df_olist_orders = pd.read_csv(
    csv_lookup["olist_orders_dataset.csv"]
)

df_olist_order_payments = pd.read_csv(
    csv_lookup["olist_order_payments_dataset.csv"]
)

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [5]:
# Initial exploration: shape, duplicate count and missing values.

datasets = {
    "customers": df_olist_customers,
    "sellers": df_olist_sellers,
    "reviews": df_olist_order_reviews,
    "order_items": df_olist_order_items,
    "products": df_olist_products,
    "geolocation": df_olist_geolocation,
    "category_translation": df_product_category_name_translation,
    "orders": df_olist_orders,
    "payments": df_olist_order_payments
}

exploration = []

for name, dataframe in datasets.items():
    exploration.append({
        "dataset": name,
        "rows": dataframe.shape[0],
        "columns": dataframe.shape[1],
        "duplicate_rows": dataframe.duplicated().sum(),
        "missing_values": dataframe.isna().sum().sum()
    })

exploration_df = pd.DataFrame(exploration)

display(exploration_df)

,dataset,rows,columns,duplicate_rows,missing_values
0,customers,99441,5,0,0
1,sellers,3095,4,0,0
2,reviews,99224,7,0,145903
3,order_items,112650,7,0,0
4,products,32951,9,0,2448
5,geolocation,1000163,5,261831,0
6,category_translation,71,2,0,0
7,orders,99441,8,0,4908
8,payments,103886,5,0,0


In [6]:
# Preview the customers table.

display(df_olist_customers.head())

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


### Initial observations

The dataset is relational: information about customers, orders, payments, products, reviews, sellers and geolocation is stored in separate tables.

Important relationships include:

- `olist_orders.customer_id` → `olist_customers.customer_id`
- `olist_order_items.order_id` → `olist_orders.order_id`
- `olist_order_items.product_id` → `olist_products_dataset.product_id`
- `olist_order_items.seller_id` → `olist_sellers.seller_id`
- `olist_order_payments.order_id` → `olist_orders.order_id`
- `olist_order_reviews.order_id` → `olist_orders.order_id`

Because some orders can contain multiple items and multiple payment rows, aggregation must be done carefully to avoid accidentally duplicating revenue.

## 2. Create SQLite Database and Export DataFrames

In [7]:
# Create an in-memory SQLite database using SQLAlchemy.

engine = create_engine("sqlite:///:memory:", echo=False)

# Export every DataFrame as a SQL table.
# index=False avoids creating an unnecessary Pandas index column.

df_olist_customers.to_sql(
    "olist_customers",
    con=engine,
    index=False,
    if_exists="replace"
)

df_olist_sellers.to_sql(
    "olist_sellers",
    con=engine,
    index=False,
    if_exists="replace"
)

df_olist_order_reviews.to_sql(
    "olist_order_reviews",
    con=engine,
    index=False,
    if_exists="replace"
)

df_olist_order_items.to_sql(
    "olist_order_items",
    con=engine,
    index=False,
    if_exists="replace"
)

df_olist_products.to_sql(
    "olist_products_dataset",
    con=engine,
    index=False,
    if_exists="replace"
)

df_olist_geolocation.to_sql(
    "olist_geolocation",
    con=engine,
    index=False,
    if_exists="replace"
)

df_product_category_name_translation.to_sql(
    "product_category_name_translation",
    con=engine,
    index=False,
    if_exists="replace"
)

df_olist_orders.to_sql(
    "olist_orders",
    con=engine,
    index=False,
    if_exists="replace"
)

df_olist_order_payments.to_sql(
    "olist_order_payments",
    con=engine,
    index=False,
    if_exists="replace"
)

print("All DataFrames were exported to SQLite.")

All DataFrames were exported to SQLite.


In [8]:
# Test the connection by selecting five rows from olist_customers.

sql = '''
SELECT *
FROM olist_customers
LIMIT 5;
'''

df_sql = pd.read_sql_query(sql, con=engine)

display(df_sql)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


## 3. Query 1 — January 2018 Orders with a 5 Review Score

In [9]:
# Count January 2018 orders, count those with a review score of 5,
# and calculate the percentage of January orders receiving five stars.

sql_query_1 = '''
WITH jan_2018_orders AS (
    SELECT order_id
    FROM olist_orders
    WHERE strftime('%Y-%m', order_purchase_timestamp) = '2018-01'
),

review_per_order AS (
    SELECT
        order_id,
        MAX(review_score) AS review_score
    FROM olist_order_reviews
    GROUP BY order_id
)

SELECT
    COUNT(*) AS jan_2018_orders,

    SUM(
        CASE
            WHEN r.review_score = 5 THEN 1
            ELSE 0
        END
    ) AS five_star_orders,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN r.review_score = 5 THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS five_star_percentage

FROM jan_2018_orders j

LEFT JOIN review_per_order r
    ON j.order_id = r.order_id;
'''

query_1_result = pd.read_sql_query(
    sql_query_1,
    con=engine
)

display(query_1_result)

,jan_2018_orders,five_star_orders,five_star_percentage
0,7269,4077,56.09


### Query 1 insight

The supplied dataset contains **7,269 orders purchased in January 2018**. Of those, **4,077 received a five-star review**, corresponding to approximately **56.09%** of January 2018 orders.

A review-level table can sometimes contain more than one record for an order, so the query first creates one review score per order before calculating the percentage.

## 4. Query 2 — Customer Purchase Trend Year-on-Year

In [10]:
# Analyze order and unique-customer growth by year.
#
# LAG() retrieves the value from the previous year so that
# we can calculate the year-on-year growth rate.

sql_query_2 = '''
WITH yearly AS (
    SELECT
        CAST(
            strftime('%Y', o.order_purchase_timestamp)
            AS INTEGER
        ) AS year,

        COUNT(DISTINCT o.order_id) AS orders,

        COUNT(
            DISTINCT c.customer_unique_id
        ) AS unique_customers

    FROM olist_orders o

    JOIN olist_customers c
        ON o.customer_id = c.customer_id

    WHERE strftime(
        '%Y',
        o.order_purchase_timestamp
    ) IN ('2016', '2017', '2018')

    GROUP BY year
),

with_previous_year AS (
    SELECT
        year,
        orders,
        unique_customers,

        LAG(orders)
            OVER (ORDER BY year)
            AS previous_year_orders,

        LAG(unique_customers)
            OVER (ORDER BY year)
            AS previous_year_customers

    FROM yearly
)

SELECT
    year,
    orders,
    unique_customers,
    previous_year_orders,

    ROUND(
        100.0 *
        (orders - previous_year_orders)
        / previous_year_orders,
        2
    ) AS order_yoy_growth_pct,

    ROUND(
        100.0 *
        (unique_customers - previous_year_customers)
        / previous_year_customers,
        2
    ) AS customer_yoy_growth_pct

FROM with_previous_year

ORDER BY year;
'''

query_2_result = pd.read_sql_query(
    sql_query_2,
    con=engine
)

display(query_2_result)

,year,orders,unique_customers,previous_year_orders,order_yoy_growth_pct,customer_yoy_growth_pct
0,2016,329,326,NaN,NaN,NaN
1,2017,45101,43713,329.00,"13,608.51","13,308.90"
2,2018,54011,52749,"45,101.00",19.76,20.67


### Query 2 insight

The platform grew very strongly after its small 2016 starting base.

- **2016:** 329 orders
- **2017:** 45,101 orders
- **2018:** 54,011 orders

The jump from 2016 to 2017 is unusually large because 2016 contains only the early portion of the dataset. From 2017 to 2018, orders increased by approximately **19.76%**, while unique customers increased by approximately **20.67%**.

## 5. Query 3 — Average Order Value of Customers

In [11]:
# First aggregate payment rows to order level.
# Then calculate the average order value for each unique customer.

sql_query_3 = '''
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_unique_id,
        SUM(p.payment_value) AS order_value

    FROM olist_orders o

    JOIN olist_customers c
        ON o.customer_id = c.customer_id

    JOIN olist_order_payments p
        ON o.order_id = p.order_id

    GROUP BY
        o.order_id,
        c.customer_unique_id
)

SELECT
    customer_unique_id,

    COUNT(*) AS number_of_orders,

    ROUND(
        AVG(order_value),
        2
    ) AS average_order_value,

    ROUND(
        SUM(order_value),
        2
    ) AS total_spent

FROM order_totals

GROUP BY customer_unique_id

ORDER BY average_order_value DESC

LIMIT 20;
'''

query_3_result = pd.read_sql_query(
    sql_query_3,
    con=engine
)

display(query_3_result)

,customer_unique_id,number_of_orders,average_order_value,total_spent
0,0a0a92112bd4c708ca5fde585afaa872,1,"13,664.08","13,664.08"
1,763c8b1c9c68a0229c42c9fc6f662b93,1,"7,274.88","7,274.88"
2,dc4802a71eae9be1dd28f5d788ceb526,1,"6,929.31","6,929.31"
3,459bef486812aa25204be022145caa62,1,"6,922.21","6,922.21"
4,ff4159b92c40ebe40454e3e6a7c35ed6,1,"6,726.66","6,726.66"
5,4007669dec559734d6f53e029e360987,1,"6,081.54","6,081.54"
6,5d0a2980b292d049061542014e8960bf,1,"4,809.44","4,809.44"
7,eebb5dda148d3893cdaf5b5ca3040ccb,1,"4,764.34","4,764.34"
8,48e1ac109decbb87765a3eade6854098,1,"4,681.78","4,681.78"
9,edde2314c6c30e864a128ac95d6b2112,1,"4,513.32","4,513.32"


### Query 3 insight

This query calculates **Average Order Value (AOV)** at the customer level rather than averaging individual payment rows. This distinction matters because one order can contain multiple payment records.

The output is sorted by AOV so DataSearch can identify customers associated with especially high-value orders.

## 6. Query 4 — Top 5 Cities by Revenue, 2016–2018

In [12]:
# Aggregate payment values to order level and then calculate
# total revenue by the customer's city.

sql_query_4 = '''
WITH order_revenue AS (
    SELECT
        o.order_id,
        c.customer_city,
        SUM(p.payment_value) AS revenue

    FROM olist_orders o

    JOIN olist_customers c
        ON o.customer_id = c.customer_id

    JOIN olist_order_payments p
        ON o.order_id = p.order_id

    WHERE strftime(
        '%Y',
        o.order_purchase_timestamp
    ) BETWEEN '2016' AND '2018'

    GROUP BY
        o.order_id,
        c.customer_city
)

SELECT
    customer_city AS city,

    COUNT(*) AS orders,

    ROUND(
        SUM(revenue),
        2
    ) AS total_revenue

FROM order_revenue

GROUP BY customer_city

ORDER BY total_revenue DESC

LIMIT 5;
'''

query_4_result = pd.read_sql_query(
    sql_query_4,
    con=engine
)

display(query_4_result)

,city,orders,total_revenue
0,sao paulo,15540,"2,203,373.09"
1,rio de janeiro,6882,"1,161,927.36"
2,belo horizonte,2773,"421,765.12"
3,brasilia,2131,"354,216.78"
4,curitiba,1521,"247,392.48"


### Query 4 insight

The five highest-revenue customer cities are:

1. **São Paulo** — approximately 2.20M
2. **Rio de Janeiro** — approximately 1.16M
3. **Belo Horizonte** — approximately 421.8K
4. **Brasília** — approximately 354.2K
5. **Curitiba** — approximately 247.4K

São Paulo is the clear revenue leader in the dataset.

## 7. Query 5 — State-Wise Revenue, 2016–2018

In [13]:
sql_query_5 = '''
WITH order_revenue AS (
    SELECT
        o.order_id,
        c.customer_state,
        SUM(p.payment_value) AS revenue

    FROM olist_orders o

    JOIN olist_customers c
        ON o.customer_id = c.customer_id

    JOIN olist_order_payments p
        ON o.order_id = p.order_id

    WHERE strftime(
        '%Y',
        o.order_purchase_timestamp
    ) BETWEEN '2016' AND '2018'

    GROUP BY
        o.order_id,
        c.customer_state
)

SELECT
    customer_state AS state,

    COUNT(*) AS orders,

    ROUND(
        SUM(revenue),
        2
    ) AS total_revenue,

    ROUND(
        AVG(revenue),
        2
    ) AS avg_order_revenue

FROM order_revenue

GROUP BY customer_state

ORDER BY total_revenue DESC;
'''

query_5_result = pd.read_sql_query(
    sql_query_5,
    con=engine
)

display(query_5_result)

,state,orders,total_revenue,avg_order_revenue
0,SP,41745,"5,998,226.96",143.69
1,RJ,12852,"2,144,379.69",166.85
2,MG,11635,"1,872,257.26",160.92
3,RS,5466,"890,898.54",162.99
4,PR,5045,"811,156.38",160.78
5,SC,3637,"623,086.43",171.32
6,BA,3380,"616,645.82",182.44
7,DF,2140,"355,141.08",165.95
8,GO,2020,"350,092.31",173.31
9,ES,2033,"325,967.55",160.34


### Query 5 insight

**São Paulo (SP)** generates the most revenue by a large margin, followed by **Rio de Janeiro (RJ)** and **Minas Gerais (MG)**.

The table also includes average order revenue, making it possible to distinguish states with high total revenue because of volume from states with relatively high spending per order.

## 8. Query 6 — Top Successful Sellers

In [14]:
# Seller success is evaluated with several metrics:
#
# - number of goods sold
# - item revenue
# - distinct customers
# - number of orders receiving five-star reviews
#
# Reviews are first reduced to one score per order to avoid duplication.

sql_query_6 = '''
WITH review_per_order AS (
    SELECT
        order_id,
        MAX(review_score) AS review_score
    FROM olist_order_reviews
    GROUP BY order_id
),

seller_stats AS (
    SELECT
        oi.seller_id,

        COUNT(*) AS goods_sold,

        ROUND(
            SUM(oi.price),
            2
        ) AS revenue,

        COUNT(
            DISTINCT c.customer_unique_id
        ) AS customer_count,

        COUNT(
            DISTINCT CASE
                WHEN r.review_score = 5
                THEN oi.order_id
            END
        ) AS five_star_orders

    FROM olist_order_items oi

    JOIN olist_orders o
        ON oi.order_id = o.order_id

    JOIN olist_customers c
        ON o.customer_id = c.customer_id

    LEFT JOIN review_per_order r
        ON oi.order_id = r.order_id

    GROUP BY oi.seller_id
)

SELECT
    seller_id,
    goods_sold,
    revenue,
    customer_count,
    five_star_orders,

    ROUND(
        100.0 * five_star_orders
        / NULLIF(customer_count, 0),
        2
    ) AS five_star_orders_per_100_customers

FROM seller_stats

ORDER BY revenue DESC

LIMIT 10;
'''

query_6_result = pd.read_sql_query(
    sql_query_6,
    con=engine
)

display(query_6_result)

,seller_id,goods_sold,revenue,customer_count,five_star_orders,five_star_orders_per_100_customers
0,4869f7a5dfa277a7dca6462dcf3b52b2,1156,"229,472.63",1124,671,59.70
1,53243585a1d6dc2643021fd1853d8905,410,"222,776.05",349,200,57.31
2,4a3ca9315b744ce9f8e9374361493884,1987,"200,472.92",1792,865,48.27
3,fa1c13f2614d7b5c4749cbc52fecda94,586,"194,042.03",581,393,67.64
4,7c67e1448b00f6e969d365cea6b010ab,1364,"187,923.89",971,343,35.32
5,7e93a43ef30c4f03f38b393420bc753a,340,"176,431.87",336,211,62.80
6,da8622b14eb17ae2831f4ac5b9dab84a,1551,"160,236.57",1275,771,60.47
7,7a67c85e85bb2ce8582c35f2203ad736,1171,"141,745.53",1155,710,61.47
8,1025f0e2d44d7041d6cf58b6550e0bfa,1428,"138,968.55",899,502,55.84
9,955fee9216a65b617aa5c0531780ce60,1499,"135,171.70",1282,719,56.08


### Query 6 insight

Seller performance should not be judged with only one measure. A seller may lead in revenue but not necessarily in customer volume or five-star reviews.

This query therefore combines:

- **sales volume**
- **revenue**
- **customer reach**
- **five-star satisfaction**

The table is ordered by revenue, but the additional columns make the comparison more complete.

## 9. Query 7 — Delivery Success Rate Across States

In [15]:
# We calculate two useful delivery KPIs:
#
# 1. Delivery success rate:
#    percentage of all orders whose final status is "delivered"
#
# 2. On-time rate:
#    percentage of delivered orders that arrived by the estimated date

sql_query_7 = '''
SELECT
    c.customer_state AS state,

    COUNT(*) AS total_orders,

    SUM(
        CASE
            WHEN o.order_status = 'delivered'
            THEN 1
            ELSE 0
        END
    ) AS delivered_orders,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN o.order_status = 'delivered'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS delivery_success_rate_pct,

    SUM(
        CASE
            WHEN o.order_status = 'delivered'
             AND o.order_delivered_customer_date
                 <= o.order_estimated_delivery_date
            THEN 1
            ELSE 0
        END
    ) AS on_time_deliveries,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN o.order_status = 'delivered'
                 AND o.order_delivered_customer_date
                     <= o.order_estimated_delivery_date
                THEN 1
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN o.order_status = 'delivered'
                    THEN 1
                    ELSE 0
                END
            ),
            0
        ),
        2
    ) AS on_time_rate_among_delivered_pct

FROM olist_orders o

JOIN olist_customers c
    ON o.customer_id = c.customer_id

GROUP BY c.customer_state

ORDER BY
    delivery_success_rate_pct DESC,
    total_orders DESC;
'''

query_7_result = pd.read_sql_query(
    sql_query_7,
    con=engine
)

display(query_7_result)

,state,total_orders,delivered_orders,delivery_success_rate_pct,on_time_deliveries,on_time_rate_among_delivered_pct
0,AC,81,80,98.77,77,96.25
1,AP,68,67,98.53,64,95.52
2,ES,2033,1995,98.13,1751,87.77
3,MS,715,701,98.04,620,88.45
4,AM,148,145,97.97,139,95.86
5,TO,280,274,97.86,239,87.23
6,RS,5466,5345,97.79,4962,92.83
7,RN,485,474,97.73,423,89.24
8,MT,907,886,97.68,826,93.23
9,MG,11635,11354,97.58,10717,94.39


### Query 7 insight

The first delivery KPI measures whether an order was successfully completed. The second measures **timeliness** among successfully delivered orders.

Keeping both indicators is useful because a state can have a high completion rate while still experiencing delays.

## 10. Query 8 — Preferred Payment Method by Product Category

In [16]:
# Determine the most frequently used payment type for every product category.
#
# Category names are translated to English when a translation exists.

sql_query_8 = '''
WITH category_payment AS (
    SELECT DISTINCT
        oi.order_id,

        COALESCE(
            t.product_category_name_english,
            pr.product_category_name,
            'unknown'
        ) AS category,

        p.payment_type

    FROM olist_order_items oi

    JOIN olist_products_dataset pr
        ON oi.product_id = pr.product_id

    LEFT JOIN product_category_name_translation t
        ON pr.product_category_name
         = t.product_category_name

    JOIN olist_order_payments p
        ON oi.order_id = p.order_id

    WHERE p.payment_type IS NOT NULL
      AND p.payment_type <> 'not_defined'
),

payment_counts AS (
    SELECT
        category,
        payment_type,
        COUNT(DISTINCT order_id)
            AS orders_using_payment

    FROM category_payment

    GROUP BY
        category,
        payment_type
),

ranked AS (
    SELECT
        category,
        payment_type,
        orders_using_payment,

        RANK() OVER (
            PARTITION BY category
            ORDER BY orders_using_payment DESC
        ) AS payment_rank

    FROM payment_counts
)

SELECT
    category,

    payment_type
        AS preferred_payment_type,

    orders_using_payment

FROM ranked

WHERE payment_rank = 1

ORDER BY
    orders_using_payment DESC,
    category;
'''

query_8_result = pd.read_sql_query(
    sql_query_8,
    con=engine
)

display(query_8_result)

,category,preferred_payment_type,orders_using_payment
0,bed_bath_table,credit_card,7540
1,health_beauty,credit_card,6874
2,sports_leisure,credit_card,5904
3,furniture_decor,credit_card,4919
4,computers_accessories,credit_card,4737
...,...,...,...
70,portateis_cozinha_e_preparadores_de_alimentos,credit_card,8
71,fashion_childrens_clothes,credit_card,5
72,pc_gamer,credit_card,5
73,security_and_services,boleto,1


### Query 8 insight

Credit card is the most common preferred payment type across the major product categories in this dataset.

Using `RANK()` inside each category allows the query to identify the most frequently used payment method separately for every category rather than selecting one payment type for the entire platform.

## 11. Query 9 — Distance Between Seller and Customer Cities

For this exercise, the distance is estimated between the **seller's ZIP-code centroid** and the **customer's ZIP-code centroid**.

Because the geolocation table contains repeated coordinates for the same ZIP prefix, we first calculate an average latitude and longitude for each ZIP prefix.

The query then applies the **Haversine formula** to estimate straight-line distance over the Earth's surface.

> Note: this is geographic distance, not driving distance.

In [17]:
sql_query_9 = '''
WITH geo AS (
    SELECT
        geolocation_zip_code_prefix
            AS zip_code_prefix,

        AVG(geolocation_lat) AS lat,
        AVG(geolocation_lng) AS lng

    FROM olist_geolocation

    GROUP BY geolocation_zip_code_prefix
),

routes AS (
    SELECT DISTINCT
        oi.order_id,

        c.customer_city,
        c.customer_state,

        s.seller_city,
        s.seller_state,

        cg.lat AS customer_lat,
        cg.lng AS customer_lng,

        sg.lat AS seller_lat,
        sg.lng AS seller_lng

    FROM olist_order_items oi

    JOIN olist_orders o
        ON oi.order_id = o.order_id

    JOIN olist_customers c
        ON o.customer_id = c.customer_id

    JOIN olist_sellers s
        ON oi.seller_id = s.seller_id

    LEFT JOIN geo cg
        ON c.customer_zip_code_prefix
         = cg.zip_code_prefix

    LEFT JOIN geo sg
        ON s.seller_zip_code_prefix
         = sg.zip_code_prefix

    WHERE cg.lat IS NOT NULL
      AND sg.lat IS NOT NULL
),

distances AS (
    SELECT
        order_id,
        customer_city,
        customer_state,
        seller_city,
        seller_state,

        6371.0 * 2 * ASIN(
            SQRT(
                POWER(
                    SIN(
                        RADIANS(
                            customer_lat - seller_lat
                        ) / 2
                    ),
                    2
                )
                +
                COS(RADIANS(seller_lat))
                *
                COS(RADIANS(customer_lat))
                *
                POWER(
                    SIN(
                        RADIANS(
                            customer_lng - seller_lng
                        ) / 2
                    ),
                    2
                )
            )
        ) AS distance_km

    FROM routes
)

SELECT
    seller_city || ' - ' || seller_state
        AS seller_location,

    customer_city || ' - ' || customer_state
        AS customer_location,

    COUNT(*) AS orders_on_route,

    ROUND(
        AVG(distance_km),
        2
    ) AS avg_distance_km

FROM distances

GROUP BY
    seller_city,
    seller_state,
    customer_city,
    customer_state

HAVING COUNT(*) >= 3

ORDER BY avg_distance_km DESC

LIMIT 20;
'''

query_9_result = pd.read_sql_query(
    sql_query_9,
    con=engine
)

display(query_9_result)

,seller_location,customer_location,orders_on_route,avg_distance_km
0,sao paulo - SP,boa vista - RR,16,"3,307.90"
1,campinas - SP,boa vista - RR,3,"3,219.10"
2,porto alegre - RS,belem - PA,6,"3,191.75"
3,sao luis - MA,porto alegre - RS,5,"3,149.59"
4,porto alegre - RS,sao luis - MA,3,"3,142.17"
5,recife - PE,porto alegre - RS,3,"2,973.37"
6,porto alegre - RS,teresina - PI,4,"2,907.27"
7,rio de janeiro - RJ,manaus - AM,3,"2,837.74"
8,blumenau - SC,belem - PA,3,"2,835.27"
9,sao luis - MA,florianopolis - SC,12,"2,821.78"


### Query 9 insight

The longest recurring seller-to-customer routes span several thousand kilometers, which reflects Brazil's large geographic size.

For example, the supplied data shows routes such as **São Paulo (SP) → Boa Vista (RR)** at roughly **3,308 km** on average.

Long-distance routes can be useful for studying freight cost, delivery time and logistics performance.

## Final Conclusions

### Key findings

- The marketplace expanded dramatically after its small 2016 starting base.
- From 2017 to 2018, order volume continued to grow by roughly **20%**.
- More than half of January 2018 orders received a **five-star review**.
- **São Paulo** is the strongest geographic market by both city and state revenue.
- Customer-level AOV provides a more meaningful spending measure than averaging individual payment rows.
- Seller success is multidimensional: revenue, unit volume, customer reach and review quality should be analyzed together.
- Delivery performance varies by state and should be evaluated using both successful-delivery and on-time-delivery rates.
- **Credit card** is the dominant payment preference across major product categories.
- Geographic distance between sellers and customers can exceed **3,000 km**, demonstrating the logistical complexity of the Brazilian market.

### SQL concepts practiced

This project uses:

- `JOIN`
- `LEFT JOIN`
- `GROUP BY`
- `CASE WHEN`
- Common Table Expressions (`WITH`)
- `COUNT(DISTINCT ...)`
- `COALESCE`
- `NULLIF`
- Window functions such as `LAG()` and `RANK()`
- Date functions with `strftime()`
- Mathematical functions for geographic distance calculations

These techniques are especially useful for working with real relational datasets where information is distributed across several linked tables.